# Ground — DEM to Contour Lines

**Library:** [`sitex`](../sitex) · **Original, fully-documented version:** [`01-ARCH-DEM Contour.ipynb`](../documentations/01-ARCH-DEM%20Contour.ipynb)

This is the thin, parameterized version of the DEM Contour workflow: a DTM GeoTIFF in,
a contour-line `GeoDataFrame` and a CAD-ready DXF out. All the logic (NaN-aware
smoothing, contour extraction, the shared local CAD origin) lives in `sitex.arch.terrain`
— this notebook is just the parameters plus the four calls that use them.

Change the `CityConfig` in the next cell to point this notebook at any geocodable place,
anywhere in the world.

## 1. Install & import

In [1]:
# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install git+https://github.com/ArchiColab/sitex.git

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install git+https://github.com/ArchiColab/sitex.git

    # Input data (read-only) — downloaded fresh each session from the workshop's GitHub release
    DATA_RELEASE_URL = "https://github.com/ArchiColab/sitex/releases/download/workshop-data-v1/Colab_Outputs.zip"
    DATA_DIR = Path("/content/Colab_Outputs")
    if not DATA_DIR.exists():
        import urllib.request, zipfile
        zip_path = Path("/content/Colab_Outputs.zip")
        urllib.request.urlretrieve(DATA_RELEASE_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)

    # Your own results — saved to your own Google Drive so they persist across sessions
    from google.colab import drive
    drive.mount("/content/gdrive")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from sitex.core.config import CityConfig
from sitex.arch import terrain

## 2. Site configuration

`CityConfig` derives the local UTM zone, a filename slug, and a shared CAD/UCS origin
from a single geocode point — the same values every phase (ARCH, ENV, network analysis)
needs, computed once instead of re-derived (and occasionally mismatched) per notebook.

In [2]:
city = CityConfig(
    place_name="Phường Pleiku, Gia Lai, Vietnam",
    local_lat=13.9833,
    local_lon=108.0000,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)

print(f"Slug        : {city.slug}")
print(f"Local EPSG  : {city.local_epsg}")
print(f"CAD origin  : {city.origin}")

Slug        : phuong_pleiku
Local EPSG  : 32649
CAD origin  : (175000.0, 1547000.0)


## 3. Generate contours

Reads `{data_dir}/dem/dtm_nasadem.tif` (from `00-Data-Acquisition_RemoteSensing.ipynb`),
reprojects to the local UTM CRS, applies NaN-aware Gaussian smoothing, and extracts
contour lines at a fixed interval.

In [3]:
INTERVAL = 2.0       # contour step, metres
SMOOTH_SIGMA = 2.0   # Gaussian smoothing radius, pixels (0 = off)

contours = terrain.generate_contours_for_city(
    city,
    interval=INTERVAL,
    smooth_sigma=SMOOTH_SIGMA,
)

print(f"{len(contours)} contour segments, {contours['ELEV'].min():.0f}–{contours['ELEV'].max():.0f} m")
contours.head()

414 contour segments, 696–796 m


,ELEV,geometry
0,696.0,"LINESTRING (182755.032 1549634.304, 182724.633..."
1,698.0,"LINESTRING (182755.032 1550163.97, 182724.633 ..."
2,698.0,"LINESTRING (182755.032 1549698.157, 182724.633..."
3,698.0,"LINESTRING (182755.032 1549069.334, 182724.633..."
4,700.0,"LINESTRING (182755.032 1550749.872, 182724.633..."


## 4. Export — GIS (Shapefile) and CAD (DXF)

The Shapefile keeps true UTM coordinates for GIS use. The DXF is shifted by the shared
`city.origin` so it lands near `(0, 0)` in CAD instead of hundreds of km away — re-apply
the same origin as the project base point / survey point (UCS) in Revit or Rhino to
georeference the model.

In [4]:
shp_path = city.data_dir / "dem" / f"contours_{city.local_epsg}.shp"
contours.to_file(shp_path)
print(f"Shapefile → {shp_path}")

dxf_path = terrain.export_contours_dxf_for_city(
    contours, city, filename=f"contours_{city.local_epsg}.dxf"
)
print(f"DXF       → {dxf_path}  (shifted by origin {city.origin})")

Shapefile → ..\..\data\dem\contours_32649.shp


C:\Users\Maddie\anaconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


DXF       → ..\..\data\dem\contours_32649.dxf  (shifted by origin (175000.0, 1547000.0))


## 5. Preview on an interactive map

In [5]:
terrain.plot_contours_folium(
    contours,
    center_lat=city.local_lat,
    center_lon=city.local_lon,
    interval=INTERVAL,
    local_epsg=city.local_epsg,
)

---
## Notes

- **`height_est`/estimation caveats and the full discussion of *why* each modelling
  choice was made** (the 20°-tolerance-style parameters, DTM vs. DSM disagreement, etc.)
  live in the original notebook and the phase overview — read those before citing this
  output in a thesis chapter.
- To model a new city: change only the `CityConfig` in Section 2. `local_epsg` and the
  CAD origin are re-derived automatically — Vietnam alone spans UTM zones 48N and 49N,
  split at 108°E, so this is not a cosmetic detail.